In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv("data.csv")


In [3]:
features = df.drop(columns=[
    "date",
    "flood_risk",
    "landslide_risk",
    "wildfire_risk",
    "glacier_melt_risk"
])

targets = df[[
    "flood_risk",
    "landslide_risk",
    "wildfire_risk",
    "glacier_melt_risk"
]]


In [4]:
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(features)
y_scaled = scaler_y.fit_transform(targets)

In [5]:
SEQ_LEN = 7

def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y_scaled, SEQ_LEN)

In [6]:
split = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

# -----------------------------
# CNN 1D Model
# -----------------------------
model = Sequential([
    Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(SEQ_LEN, X_seq.shape[2])),
    MaxPooling1D(pool_size=2),

    Conv1D(filters=128, kernel_size=2, activation='relu'),
    Flatten(),

    Dense(64, activation='relu'),
    Dense(4)
])

model.compile(optimizer='adam', loss='mse')


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.fit(X_train, y_train, epochs=100, batch_size=16)

print("CNN-1D training complete")

Epoch 1/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.0722e-04 
Epoch 2/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.3384e-04 
Epoch 3/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.2945e-04 
Epoch 4/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.4890e-04 
Epoch 5/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 4.7904e-04 
Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.9440e-04 
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.7791e-04 
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 5.9403e-04
Epoch 9/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.7177e-04 
Epoch 10/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.2042e-04 
Epoch 11/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.5616e-04 
Epoch 12/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 4.2504e-04 
Epoch 13/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 2.3479e-04 
Epoch 14/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.1550e-04 
Ep

In [10]:
model.save("cnn_1d_model.h5")

In [11]:
import joblib
joblib.dump(scaler_X, "cnn_scaler_X.pkl")
joblib.dump(scaler_y, "cnn_scaler_y.pkl")


['cnn_scaler_y.pkl']

In [12]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score
import numpy as np


In [13]:
y_pred = model.predict(X_test)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step 


In [14]:
y_pred_inv = scaler_y.inverse_transform(y_pred)
y_test_inv = scaler_y.inverse_transform(y_test)

In [15]:
rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
mae = mean_absolute_error(y_test_inv, y_pred_inv)


In [16]:
threshold = 0.6

y_pred_bin = (y_pred_inv > threshold).astype(int)
y_test_bin = (y_test_inv > threshold).astype(int)

accuracy = accuracy_score(
    y_test_bin.flatten(),
    y_pred_bin.flatten()
)

In [18]:
print("\nCNN PERFORMANCE:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Accuracy: {accuracy:.2f}")


 CNN PERFORMANCE:
RMSE: 0.1218
MAE: 0.0738
Accuracy: 90.45
